In [1]:
import argparse
import pandas as pd
from pathlib import Path

SHIFT_DIR = Path("/mnt/beegfs/msaxena4/6_Explicit-Implicit-Bias/ResultAnalysis/Summarization/SampleResults")
OUT_DIR   = SHIFT_DIR / "Tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS = ["gemma", "llama", "phi", "qwen", "mistral"]

In [2]:
# ── Load ──────────────────────────────────────────────────────────────────────
append_dfs = []
for model in MODELS:
    ap = SHIFT_DIR / f"2_SemanticShift/{model}_Append_Embeddings.csv"
    if ap.exists():
        df = pd.read_csv(ap)
        df["model"] = model
        append_dfs.append(df)
    else:
        print(f"  WARNING: {ap.name} not found, skipping")

if not append_dfs:
    print("No append CSVs found. Exiting.")
    raise SystemExit(1)

df = pd.concat(append_dfs, ignore_index=True)
sbert = df[df["embedding"] == "sbert"].copy()
sbert["mean_cosine_similarity"] = (1 - sbert["mean_cosine_distance"]).round(4)
sbert["cohens_dz"] = sbert["cohens_dz"].round(4)
sbert["p"]         = sbert["p"].round(4)
sbert["p_bh"]      = sbert["p_bh"].round(4)

COLS = ["model", "myth", "is_pair", "test", "n",
        "mean_cosine_similarity", "cohens_dz",
        "p", "p_bh", "significant"]

In [4]:
sbert.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50 entries, 0 to 129
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   test                    50 non-null     object 
 1   stat                    50 non-null     float64
 2   p                       50 non-null     float64
 3   normal                  50 non-null     bool   
 4   n                       50 non-null     int64  
 5   cohens_dz               50 non-null     float64
 6   cohens_dav              50 non-null     float64
 7   cohens_drm              0 non-null      float64
 8   hedges_g                50 non-null     float64
 9   embedding               50 non-null     object 
 10  myth                    50 non-null     object 
 11  is_pair                 50 non-null     bool   
 12  mean_cosine_distance    50 non-null     float64
 13  median_cosine_distance  50 non-null     float64
 14  p_bh                    50 non-null     float64


In [17]:
# # ── Table 1: Single myths ─────────────────────────────────────────────────────
# t1 = (
#     sbert[sbert["is_pair"] == False][COLS]
#     .sort_values(["model", "myth"])
#     .reset_index(drop=True)
# )
# out1 = OUT_DIR / f"Table1_SingleMyths_ContextAppend.csv"
# t1.to_csv(out1, index=False)
# print(f"Saved: {out1.name}")
# print(t1.to_string(index=False))


In [16]:
# # ── Table 2: Myth pairs ───────────────────────────────────────────────────────
# MYTH_TYPES = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]

# t2 = sbert[sbert["is_pair"] == True].copy()

# for myth in MYTH_TYPES:
#     t2[myth] = t2["myth"].str.contains(myth).astype(int)

# t2 = (
#     t2[MYTH_TYPES + COLS]
#     .sort_values(["model", "myth"])
#     .reset_index(drop=True)
# )
# out2 = OUT_DIR / f"Table2_MythPairs_ContextAppend.csv"
# t2.to_csv(out2, index=False)
# print(f"Saved: {out2.name}")
# print(t2.to_string(index=False))

In [21]:
# ── Combined table: single myths + pairs ─────────────────────────────────────
MYTH_TYPES = ["clothing", "victim_intoxication", "perpetrator_intoxication", "resistance"]

combined = sbert.copy()
for myth in MYTH_TYPES:
    combined[myth] = combined["myth"].str.contains(myth).astype(int)

combined = (
    combined[MYTH_TYPES + COLS]
    .sort_values(["model", "is_pair", "myth"])
    .reset_index(drop=True)
)
out = OUT_DIR / f"Table1_ContextAppend.csv"
combined.to_csv(out, index=False)
print(f"Saved: {out.name}")
print(combined.to_string(index=False))

Saved: Table1_ContextAppend.csv
 clothing  victim_intoxication  perpetrator_intoxication  resistance   model                                         myth  is_pair                 test    n  mean_cosine_similarity  cohens_dz   p  p_bh  significant
        1                    0                         0           0   gemma                                     clothing    False wilcoxon_signed_rank 2360                  0.8157    -2.7919 0.0   0.0         True
        0                    0                         1           0   gemma                     perpetrator_intoxication    False wilcoxon_signed_rank 2344                  0.8258    -2.8978 0.0   0.0         True
        0                    0                         0           1   gemma                                   resistance    False wilcoxon_signed_rank 1416                  0.8367    -2.6290 0.0   0.0         True
        0                    1                         0           0   gemma                          victim